# code-server on Colab

Run all. With `ZROK_TOKEN` + `ZROK_NAME` you get a **fixed** URL
(`https://<name>.shares.zrok.io`). Without them it falls back to a Cloudflare
quick tunnel, which needs no account but changes hostname every start.

> Colab's FAQ does not permit remote-IDE use and Colab warns when it detects one.

## 1. Scripts

In [ ]:
import pathlib, urllib.request

REF  = "25a09ee4"    # pin a commit SHA
REPO = "recursive-factory/code-server"
BASE = pathlib.Path("/content/cs")

# ?v=<sha> is a cache-buster: raw.githubusercontent.com (or a Colab egress proxy)
# can otherwise serve an older copy of a path you just pushed, silently.
for rel in ["lib/common.sh", "install.sh", "serve.sh", "supervise.sh", "stop.sh"]:
    f = BASE / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(urllib.request.urlopen(
        f"https://raw.githubusercontent.com/{REPO}/{REF}/colab/{rel}?v={REF}",
        timeout=30).read().decode())
    if f.suffix == ".sh":
        f.chmod(0o755)

srv = (BASE / "serve.sh").read_text()
print("fetched ->", BASE, "at", REF)
print("  stale-share release:", "zrok2 delete share" in srv)


## 2. Configure

In [ ]:
import os

# Empty value -> real environment variable, or a Colab Secret of the same name.
# ZROK_* give a fixed URL; without them a rotating cloudflared quick tunnel.
ZROK_TOKEN = ""
ZROK_NAME  = ""
ZROK_AUTOCLAIM = True   # if the name is taken, claim a stable "<name>-<hash>" variant

TUNNEL        = ""     # "" = auto | zrok | cloudflared | none
CS_PORT       = ""     # "" = first free port in CS_PORT_RANGE
CS_PORT_RANGE = ""     # "" = 3000-5000
IDLE          = 0      # 0 = never stop for idleness
MAX_RUNTIME   = 39600  # hard stop at 11 h

def _pick(cell, name):
    """Secrets are not env vars: userdata.get is the only access."""
    if cell.strip() or os.environ.get(name, "").strip():
        return cell.strip() or os.environ[name].strip()
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception as e:
        if "access" in str(e).lower():
            print(f"[secret] grant notebook access for '{name}' (key icon)")
        return ""

os.environ["ZROK_TOKEN"] = _pick(ZROK_TOKEN, "ZROK_TOKEN")
os.environ["ZROK_NAME"]  = _pick(ZROK_NAME,  "ZROK_NAME")
os.environ["ZROK_AUTOCLAIM"] = "1" if ZROK_AUTOCLAIM else "0"
os.environ["ZROK_RELEASE"] = "1" if ZROK_RELEASE else "0"
os.environ.update(CS_TUNNEL=TUNNEL, CS_PORT=str(CS_PORT),
                  CS_IDLE_TIMEOUT=str(IDLE), CS_MAX_RUNTIME=str(MAX_RUNTIME))
if CS_PORT_RANGE:
    os.environ["CS_PORT_RANGE"] = CS_PORT_RANGE

_name = os.environ["ZROK_NAME"]
print("url:", f"https://{_name}.shares.zrok.io" if _name else "cloudflared (rotating)",
      "| autoclaim:", ZROK_AUTOCLAIM, "| idle:", IDLE or "off")


## 3. Install

In [ ]:
import subprocess
subprocess.run(["bash", f"{BASE}/install.sh"], check=True)

## 4. Start (detached)

In [ ]:
import os, pathlib, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
STATE.mkdir(parents=True, exist_ok=True)

# stop.sh retires any previous supervisor and reclaims the port, so this is re-runnable.
subprocess.run(["bash", f"{BASE}/stop.sh"], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
(STATE / "credentials.txt").unlink(missing_ok=True)

with open(STATE / "supervise.out", "w") as fh:
    proc = subprocess.Popen(["bash", f"{BASE}/supervise.sh"], stdout=fh,
                            stderr=subprocess.STDOUT, start_new_session=True)
(STATE / "supervise.pid").write_text(str(proc.pid))
print("supervisor", proc.pid, "| log:", STATE / "supervise.out")

## 5. URL

Waits at most 30 s; the stack keeps running in the background, so re-run this
cell rather than blocking.

In [ ]:
import os, time, pathlib, subprocess
from IPython.display import display, clear_output, HTML

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
creds, log = STATE / "credentials.txt", STATE / "supervise.out"

for _ in range(30):
    info = {}
    if creds.exists():
        for line in creds.read_text().splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                info[key] = value
    clear_output(wait=True)
    if info.get("url"):
        display(HTML(f'<p style="font-size:17px"><b>code-server</b> '
                     f'<a href="{info["url"]}" target="_blank">{info["url"]}</a></p>'
                     f'<p><b>password</b> <code>{info.get("password")}</code></p>'))
        print("tunnel:", info.get("tunnel"), "| port:", info.get("port"))
        break
    print("\n".join(log.read_text().splitlines()[-6:]) if log.exists() else "(starting)")
    time.sleep(1)
else:
    print("not up yet — re-run this cell.\nlog:", log)

## 6. Stop (opt-in)

In [ ]:
STOP = False        # set True, then run THIS cell alone: "Run all" executes it too

import os, pathlib, signal, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
if not STOP:
    print("skipped — stopping is opt-in so Run all cannot kill the stack.")
else:
    pid = STATE / "supervise.pid"
    if pid.exists():
        try:
            os.killpg(os.getpgid(int(pid.read_text())), signal.SIGTERM)
        except (ValueError, ProcessLookupError):
            pass
        pid.unlink(missing_ok=True)
    subprocess.run(["bash", f"{BASE}/stop.sh"], check=False)

## Troubleshooting

| symptom | cause |
|---|---|
| `zrok name ... is taken by another zrok user` | public names are global — reserve another with `zrok2 create name -n public <name>` |
| `zrok enable failed` | bad/expired `ZROK_TOKEN` |
| URL changes every start | no zrok name configured, so the cloudflared quick tunnel is used |
| blank page | quick-tunnel interstitial not clicked through |
| `port ... is busy` | orphan; `bash colab/stop.sh` or re-run cell 4 |

Logs: `/content/.code-server/{supervise.out,zrok.log,cloudflared.log}`